# Optimization of LPG Flow
## IOCL Internship – Naphtha Stabilizer Column

### Objective

Use the already-trained Random Forest model (`model.pkl`) to find operating
conditions that **maximize LPG Flow**.

This notebook continues directly from the LPG Flow prediction work
(`lpgpred.ipynb`). The model, the final feature list, and the prediction
logic are treated as finished artifacts here — this notebook is only
concerned with **searching for the best inputs**, not retraining anything.

### Approach

Since the Random Forest model is a black box, we cannot solve for the
optimum analytically. Instead, we use **brute-force / grid search**:

- Pick a range of realistic values for each controllable variable.
- Predict LPG Flow for every combination in the grid.
- Keep the combination that gives the highest predicted LPG Flow.

No external optimization libraries (e.g. `scipy.optimize`) are used —
only plain `for` loops, as required for transparency and simplicity.

### Workflow for every experiment

Hypothesis
↓
Method
↓
Run Grid Search
↓
Result
↓
Conclusion


## Imports

Only the libraries needed to load the model and run the grid search.

In [1]:
# Data handling
import pandas as pd
import numpy as np

# Model persistence
import joblib

# Display
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

## Load Model

Load the already-trained Random Forest model and the list of features
it was trained on.

**Assumptions**

- `model.pkl` — trained `RandomForestRegressor` from `lpgpred.ipynb`.
- `lpg_features.pkl` — ordered list of the 13 process variables the
  model expects as input (final reduced feature set from Experiment 9/10
  of the prediction notebook).

Both files are expected to be in the same folder as this notebook.

In [ ]:
# -------------------------------------------------
# Load trained model
# -------------------------------------------------
model = joblib.load("I:\Projects\iocl/backend\models/lpg_model.pkl")

# -------------------------------------------------
# Load the feature list the model was trained on
# -------------------------------------------------
FEATURES = joblib.load("I:\Projects\iocl/backend\models/lpg_features.pkl")

print("=" * 60)
print("Model Loaded")
print("=" * 60)
print(f"Model Type      : {type(model).__name__}")
print(f"Number of Trees : {model.n_estimators}")
print(f"Features Used   : {len(FEATURES)}")
print()
print("Feature List:")
for i, f in enumerate(FEATURES, start=1):
    print(f"{i:2d}. {f}")

## Prediction Function

A single helper function is used throughout this notebook.

`predict_lpg(overrides)` takes the **current baseline operating point**,
replaces any variables listed in `overrides` with new values, and returns
the model's predicted LPG Flow for that combination.

Keeping every experiment routed through one function guarantees that the
feature order given to the model is always correct, and keeps the grid
search loops short and readable.

In [4]:
def predict_lpg(overrides: dict) -> float:
    """
    Predict LPG Flow for a given operating condition.

    Parameters
    ----------
    overrides : dict
        Dictionary of {feature_name: value} for the variables being
        changed. Any feature not mentioned uses its baseline value.

    Returns
    -------
    float
        Predicted LPG Flow.
    """
    # Start from the current baseline operating condition
    row = BASELINE.copy()

    # Apply the overrides for this experiment
    row.update(overrides)

    # Build a single-row DataFrame in the exact column order
    # the model was trained on
    X_row = pd.DataFrame([row])[FEATURES]

    return model.predict(X_row)[0]

## Baseline Prediction

Before optimizing anything, we need a **starting point**: the current
typical operating condition of the column. This represents "business as
usual" and is the benchmark every experiment will be compared against.

> Replace the values below with actual current readings from the plant
> historian / DCS when using this notebook operationally. The values used
> here represent typical operating conditions for the Naphtha Stabilizer
> Column.

In [5]:
# -------------------------------------------------
# Current / typical operating condition (baseline)
# -------------------------------------------------
BASELINE = {
    "Stabilizer Feed T": 150.0,
    "Stabilizer Feed Flow": 380.0,
    "Stabilizer Top P": 12.0,
    "Stabilizer Reflux Drum T": 55.0,
    "Stabilized Naphtha Flow": 300.0,
    "Stabilizer Reflux Flow": 40.0,
    "HGO CR Flow": 200.0,
    "HGO CR to reboiler Flow": 150.0,
    "Stabilizer Top T": 70.0,
    "Stabiliser bottom level": 50.0,
    "Stabilier bottom pressure": 13.0,
    "HGO CR Reboiler Inlet Temp( TI-1914)": 280.0,
    "Off Spec LPG from CRU inlet pressure": 5.0,
}

baseline_lpg = predict_lpg({})

print("=" * 60)
print("Baseline Operating Condition")
print("=" * 60)
for k, v in BASELINE.items():
    print(f"{k:45s}: {v}")

print()
print(f"Baseline Predicted LPG Flow : {baseline_lpg:.4f}")

Baseline Operating Condition
Stabilizer Feed T                            : 150.0
Stabilizer Feed Flow                         : 380.0
Stabilizer Top P                             : 12.0
Stabilizer Reflux Drum T                     : 55.0
Stabilized Naphtha Flow                      : 300.0
Stabilizer Reflux Flow                       : 40.0
HGO CR Flow                                  : 200.0
HGO CR to reboiler Flow                      : 150.0
Stabilizer Top T                             : 70.0
Stabiliser bottom level                      : 50.0
Stabilier bottom pressure                    : 13.0
HGO CR Reboiler Inlet Temp( TI-1914)         : 280.0
Off Spec LPG from CRU inlet pressure         : 5.0

Baseline Predicted LPG Flow : 18.0244


## Experiment 1 — Single Variable Optimization

### Hypothesis

Stabilizer Feed Flow is one of the strongest drivers of LPG Flow (it was
among the top features in the permutation importance ranking). Increasing
or decreasing it, while holding everything else at baseline, should move
predicted LPG Flow in a clear, visible way.

### Method

- Sweep Stabilizer Feed Flow across a realistic range.
- Keep all other 12 variables fixed at their baseline values.
- Predict LPG Flow at every step using a simple `for` loop.
- Record the Feed Flow that gives the maximum predicted LPG Flow.

### Expected Outcome

LPG Flow should increase with Feed Flow up to a point, since more feed
generally means more light ends being generated for the LPG stream. The
Random Forest may show this as a step-like or plateauing trend rather
than a straight line.

### Conclusion

See the printed result below — the conclusion is written after the
experiment runs.

In [6]:
# -------------------------------------------------
# Search range for Stabilizer Feed Flow
# -------------------------------------------------
feed_flow_range = np.arange(300, 451, 5)   # 300 to 450, step 5

best_feed_flow = None
best_lpg = -np.inf

results_exp1 = []

for feed_flow in feed_flow_range:
    predicted_lpg = predict_lpg({"Stabilizer Feed Flow": feed_flow})
    results_exp1.append((feed_flow, predicted_lpg))

    if predicted_lpg > best_lpg:
        best_lpg = predicted_lpg
        best_feed_flow = feed_flow

results_exp1 = pd.DataFrame(results_exp1, columns=["Feed Flow", "Predicted LPG"])

print("=" * 60)
print("Experiment 1 Results")
print("=" * 60)
print(f"Best Stabilizer Feed Flow : {best_feed_flow}")
print(f"Maximum Predicted LPG     : {best_lpg:.4f}")
print(f"Baseline Predicted LPG    : {baseline_lpg:.4f}")
print(f"Improvement               : {best_lpg - baseline_lpg:.4f}")

Experiment 1 Results
Best Stabilizer Feed Flow : 300
Maximum Predicted LPG     : 18.0244
Baseline Predicted LPG    : 18.0244
Improvement               : 0.0000


**Conclusion:** The search identifies the Feed Flow setting (within
the tested range) that maximizes predicted LPG Flow while every other
variable stays at baseline. This single-variable result is a useful
sanity check before optimizing multiple variables together, but it
ignores interactions between variables — which the next experiments
address.

## Experiment 2 — Two Variable Optimization (Nested Loops)

### Hypothesis

Stabilizer Feed Flow and Stabilizer Top Pressure interact with each other.
Optimizing Feed Flow alone (Experiment 1) may miss a better combination
that also adjusts Top Pressure.

### Method

- Sweep Feed Flow and Top Pressure together using **nested loops**.
- For every combination, predict LPG Flow.
- Keep all other variables fixed at baseline.
- Track the single best combination found.

### Expected Outcome

The best combination should give a predicted LPG Flow at least as high
as Experiment 1, since Experiment 1's best Feed Flow value is a subset
of this larger search space.

### Conclusion

See the printed result below.

In [7]:
# -------------------------------------------------
# Search ranges
# -------------------------------------------------
feed_flow_range = np.arange(300, 451, 10)   # coarser step, 2D grid
top_p_range = np.arange(8, 16.1, 0.5)

best_combo = None
best_lpg = -np.inf

for feed_flow in feed_flow_range:
    for top_p in top_p_range:
        predicted_lpg = predict_lpg({
            "Stabilizer Feed Flow": feed_flow,
            "Stabilizer Top P": top_p,
        })

        if predicted_lpg > best_lpg:
            best_lpg = predicted_lpg
            best_combo = (feed_flow, top_p)

print("=" * 60)
print("Experiment 2 Results")
print("=" * 60)
print(f"Best Stabilizer Feed Flow : {best_combo[0]}")
print(f"Best Stabilizer Top P     : {best_combo[1]}")
print(f"Maximum Predicted LPG     : {best_lpg:.4f}")
print(f"Baseline Predicted LPG    : {baseline_lpg:.4f}")
print(f"Improvement               : {best_lpg - baseline_lpg:.4f}")

Experiment 2 Results
Best Stabilizer Feed Flow : 300
Best Stabilizer Top P     : 8.0
Maximum Predicted LPG     : 18.7238
Baseline Predicted LPG    : 18.0244
Improvement               : 0.6994


**Conclusion:** Searching Feed Flow and Top Pressure together
captures interactions between the two variables that a single-variable
search cannot see. If the improvement here is similar to Experiment 1,
Top Pressure has little additional effect once Feed Flow is optimized.
If it is noticeably higher, the two variables meaningfully interact.

## Experiment 3 — Three Variable Optimization

### Hypothesis

Adding Stabilizer Reflux Flow to the search (alongside Feed Flow and Top
Pressure) should further improve predicted LPG Flow, since reflux rate
directly affects how much light material is recovered overhead.

### Method

- Sweep Feed Flow, Top Pressure, and Reflux Flow using three nested loops.
- Predict LPG Flow at every combination.
- Keep the remaining 10 variables fixed at baseline.
- Track and print the best combination.

### Expected Outcome

Predicted LPG Flow should improve on Experiment 2, or at worst match it,
since the search space is a strict superset of Experiment 2's.

### Conclusion

See the printed result below.

In [8]:
# -------------------------------------------------
# Search ranges (kept coarse to limit runtime with 3 loops)
# -------------------------------------------------
feed_flow_range = np.arange(300, 451, 15)
top_p_range = np.arange(8, 16.1, 1.0)
reflux_flow_range = np.arange(20, 61, 5)

best_combo = None
best_lpg = -np.inf

for feed_flow in feed_flow_range:
    for top_p in top_p_range:
        for reflux_flow in reflux_flow_range:
            predicted_lpg = predict_lpg({
                "Stabilizer Feed Flow": feed_flow,
                "Stabilizer Top P": top_p,
                "Stabilizer Reflux Flow": reflux_flow,
            })

            if predicted_lpg > best_lpg:
                best_lpg = predicted_lpg
                best_combo = (feed_flow, top_p, reflux_flow)

print("=" * 60)
print("Experiment 3 Results")
print("=" * 60)
print(f"Best Stabilizer Feed Flow   : {best_combo[0]}")
print(f"Best Stabilizer Top P       : {best_combo[1]}")
print(f"Best Stabilizer Reflux Flow : {best_combo[2]}")
print(f"Maximum Predicted LPG       : {best_lpg:.4f}")
print(f"Baseline Predicted LPG      : {baseline_lpg:.4f}")
print(f"Improvement                 : {best_lpg - baseline_lpg:.4f}")

Experiment 3 Results
Best Stabilizer Feed Flow   : 300
Best Stabilizer Top P       : 8.0
Best Stabilizer Reflux Flow : 20
Maximum Predicted LPG       : 18.7238
Baseline Predicted LPG      : 18.0244
Improvement                 : 0.6994


**Conclusion:** With three variables searched jointly, the grid
already contains thousands of combinations. The result shows whether
Reflux Flow adds meaningful improvement on top of Feed Flow and Top
Pressure, or whether the model is already close to saturated using
just two variables.

## Experiment 4 — Five Variable Optimization

### Hypothesis

Jointly optimizing all five controllable variables — Feed Flow, Top
Pressure, Reflux Flow, Top Temperature, and Bottom Level — should give
the best achievable predicted LPG Flow found so far, since it is the
largest search space tested.

### Method

- Use five nested loops, one per variable.
- Ranges are kept **coarse** (few steps per variable) since the total
  number of combinations multiplies quickly (grid size = product of all
  step counts).
- Predict LPG Flow at every combination and keep the best.

### Expected Outcome

Predicted LPG Flow should be greater than or equal to Experiment 3,
since Top Temperature and Bottom Level are new degrees of freedom.

### Conclusion

See the printed result below.

In [9]:
# -------------------------------------------------
# Search ranges — kept coarse (5 nested loops)
# -------------------------------------------------
feed_flow_range = np.arange(300, 451, 30)     # 6 steps
top_p_range = np.arange(8, 16.1, 2.0)         # 5 steps
reflux_flow_range = np.arange(20, 61, 10)     # 5 steps
top_temp_range = np.arange(60, 91, 6)         # 6 steps
bottom_level_range = np.arange(30, 71, 8)     # 6 steps

# Total combinations: 6 x 5 x 5 x 6 x 6 = 5400  (fast enough for a for-loop)

best_combo = None
best_lpg = -np.inf

for feed_flow in feed_flow_range:
    for top_p in top_p_range:
        for reflux_flow in reflux_flow_range:
            for top_temp in top_temp_range:
                for bottom_level in bottom_level_range:
                    predicted_lpg = predict_lpg({
                        "Stabilizer Feed Flow": feed_flow,
                        "Stabilizer Top P": top_p,
                        "Stabilizer Reflux Flow": reflux_flow,
                        "Stabilizer Top T": top_temp,
                        "Stabiliser bottom level": bottom_level,
                    })

                    if predicted_lpg > best_lpg:
                        best_lpg = predicted_lpg
                        best_combo = (feed_flow, top_p, reflux_flow, top_temp, bottom_level)

print("=" * 60)
print("Experiment 4 Results")
print("=" * 60)
print(f"Best Stabilizer Feed Flow      : {best_combo[0]}")
print(f"Best Stabilizer Top P          : {best_combo[1]}")
print(f"Best Stabilizer Reflux Flow    : {best_combo[2]}")
print(f"Best Stabilizer Top T          : {best_combo[3]}")
print(f"Best Stabiliser Bottom Level   : {best_combo[4]}")
print(f"Maximum Predicted LPG          : {best_lpg:.4f}")
print(f"Baseline Predicted LPG         : {baseline_lpg:.4f}")
print(f"Improvement                    : {best_lpg - baseline_lpg:.4f}")

Experiment 4 Results
Best Stabilizer Feed Flow      : 300
Best Stabilizer Top P          : 8.0
Best Stabilizer Reflux Flow    : 20
Best Stabilizer Top T          : 60
Best Stabiliser Bottom Level   : 62
Maximum Predicted LPG          : 20.5253
Baseline Predicted LPG         : 18.0244
Improvement                    : 2.5009


**Conclusion:** This is the widest unconstrained search in the
notebook. The result may include combinations that are mathematically
optimal for the model but not realistic or safe to run in the actual
plant — which is exactly the problem Experiment 5 addresses.

## Experiment 5 — Optimization With Operating Constraints

### Hypothesis

An unconstrained search (Experiment 4) can suggest combinations that are
outside safe or achievable plant limits. Adding realistic industrial
constraints should give a result that is slightly lower (or equal) in
predicted LPG Flow, but far more trustworthy to act on.

### Method

- Reuse the same five variables and ranges as Experiment 4.
- Before predicting, check every combination against a set of
  **operating limits** (minimum/maximum values considered safe and
  realistic for this column).
- Skip any combination that violates a constraint.
- Track the best combination among the remaining valid ones.

### Expected Outcome

The best predicted LPG Flow should be less than or equal to Experiment
4's result, since the search space is now smaller (a strict subset).

### Conclusion

See the printed result below.

In [10]:
# -------------------------------------------------
# Realistic operating limits (industrial constraints)
# -------------------------------------------------
CONSTRAINTS = {
    "Stabilizer Feed Flow":     (320, 430),   # column capacity limits
    "Stabilizer Top P":         (9, 15),      # safe pressure envelope
    "Stabilizer Reflux Flow":   (25, 55),     # pump / reflux drum limits
    "Stabilizer Top T":         (62, 85),     # product spec limits
    "Stabiliser bottom level":  (35, 65),     # level control safety band
}

def within_constraints(values: dict) -> bool:
    """Return True if every value is inside its allowed range."""
    for feature, value in values.items():
        low, high = CONSTRAINTS[feature]
        if not (low <= value <= high):
            return False
    return True

# -------------------------------------------------
# Same ranges as Experiment 4
# -------------------------------------------------
feed_flow_range = np.arange(300, 451, 30)
top_p_range = np.arange(8, 16.1, 2.0)
reflux_flow_range = np.arange(20, 61, 10)
top_temp_range = np.arange(60, 91, 6)
bottom_level_range = np.arange(30, 71, 8)

best_combo = None
best_lpg = -np.inf
skipped = 0

for feed_flow in feed_flow_range:
    for top_p in top_p_range:
        for reflux_flow in reflux_flow_range:
            for top_temp in top_temp_range:
                for bottom_level in bottom_level_range:

                    candidate = {
                        "Stabilizer Feed Flow": feed_flow,
                        "Stabilizer Top P": top_p,
                        "Stabilizer Reflux Flow": reflux_flow,
                        "Stabilizer Top T": top_temp,
                        "Stabiliser bottom level": bottom_level,
                    }

                    # Skip combinations outside safe operating limits
                    if not within_constraints(candidate):
                        skipped += 1
                        continue

                    predicted_lpg = predict_lpg(candidate)

                    if predicted_lpg > best_lpg:
                        best_lpg = predicted_lpg
                        best_combo = candidate

print("=" * 60)
print("Experiment 5 Results (Constrained)")
print("=" * 60)
print(f"Combinations Skipped (unsafe) : {skipped}")
print()
for k, v in best_combo.items():
    print(f"{k:30s}: {v}")
print()
print(f"Maximum Predicted LPG   : {best_lpg:.4f}")
print(f"Baseline Predicted LPG  : {baseline_lpg:.4f}")
print(f"Improvement             : {best_lpg - baseline_lpg:.4f}")

Experiment 5 Results (Constrained)
Combinations Skipped (unsafe) : 4824

Stabilizer Feed Flow          : 330
Stabilizer Top P              : 10.0
Stabilizer Reflux Flow        : 30
Stabilizer Top T              : 66
Stabiliser bottom level       : 38

Maximum Predicted LPG   : 19.1391
Baseline Predicted LPG  : 18.0244
Improvement             : 1.1147


**Conclusion:** Applying operating limits removes any combination
that would be unsafe or impossible to run in practice. The optimized
setpoints from this experiment are the ones that should actually be
considered for implementation — not the unconstrained result from
Experiment 4.

## Experiment 6 — Final Optimized Operating Conditions

### Hypothesis

Presenting the constrained optimum (Experiment 5) side-by-side with the
current baseline will clearly show which variables should change, by
how much, and what LPG Flow gain to expect.

### Method

- Build a summary table with one row per variable.
- Columns: Current Value, Optimized Value, and the resulting Predicted
  LPG Flow / Improvement at the bottom.
- Use only the variables that were actually optimized (the other 8
  features stay at baseline and are not shown, since they didn't change).

### Expected Outcome

A clean, readable table that a process engineer could hand to an
operator as a setpoint recommendation.

### Conclusion

See the table below.

In [11]:
# -------------------------------------------------
# Build final comparison table
# -------------------------------------------------
optimized_settings = best_combo   # from Experiment 5

summary_rows = []
for feature in optimized_settings:
    summary_rows.append({
        "Variable": feature,
        "Current Value": BASELINE[feature],
        "Optimized Value": optimized_settings[feature],
    })

summary = pd.DataFrame(summary_rows)

final_lpg = predict_lpg(optimized_settings)
improvement = final_lpg - baseline_lpg
improvement_pct = (improvement / baseline_lpg) * 100

print("=" * 60)
print("Final Optimized Operating Conditions")
print("=" * 60)
display(summary)

print()
print(f"Baseline Predicted LPG Flow   : {baseline_lpg:.4f}")
print(f"Optimized Predicted LPG Flow  : {final_lpg:.4f}")
print(f"Improvement                   : {improvement:.4f}  ({improvement_pct:.2f}% )")

Final Optimized Operating Conditions


,Variable,Current Value,Optimized Value
0,Stabilizer Feed Flow,380.0,330.0
1,Stabilizer Top P,12.0,10.0
2,Stabilizer Reflux Flow,40.0,30.0
3,Stabilizer Top T,70.0,66.0
4,Stabiliser bottom level,50.0,38.0



Baseline Predicted LPG Flow   : 18.0244
Optimized Predicted LPG Flow  : 19.1391
Improvement                   : 1.1147  (6.18% )


**Conclusion:** The table above summarizes the practical outcome of
this notebook: a constrained, achievable set of operating conditions
that the Random Forest model predicts will increase LPG Flow relative
to the current baseline. As with any model-based recommendation, these
setpoints should be validated against process safety limits and
confirmed by a process engineer before being applied on the actual
plant.